# Prediction Markets Monitor — Day 13 Walkthrough

## Objective

Build a prediction-market monitor that:

1. learns from resolved historical markets,
2. benchmarks model performance against market-implied probabilities,
3. selects the strongest snapshot timing,
4. scores current live markets,
5. surfaces model-vs-market disagreements in a readable way.

## What this notebook covers

- the dataset spine
- snapshot logic
- offline evaluation summary
- best 24h model interpretation
- current Polymarket scoring
- current Kalshi scoring
- confidence / trust caveats
- final outputs and next steps

In [ ]:
# Resolve paths relative to repo root even when the notebook runs from /notebooks

from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

# Assumes notebook is in: <repo>/notebooks/day13_walkthrough.ipynb
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

print("Working directory:", Path.cwd())
print("Base directory:", BASE_DIR)

paths = {
    "markets_recent": BASE_DIR / "data/processed/markets_recent.parquet",
    "features_mid": BASE_DIR / "data/processed/features_mid_recent_history_enriched.parquet",
    "features_24h": BASE_DIR / "data/processed/features_24h_recent_history_enriched.parquet",
    "offline_summary_csv": BASE_DIR / "artifacts/outputs/offline_snapshot_summary.csv",
    "coefficients_csv": BASE_DIR / "artifacts/outputs/best_24h_coefficients.csv",
    "poly_scored": BASE_DIR / "data/processed/current_polymarket_scored_trained.parquet",
    "kalshi_scored": BASE_DIR / "data/processed/kalshi_current_scored_trained.parquet",
    "monitor_report": BASE_DIR / "reports/current_monitor_report.md",
    "final_model_report": BASE_DIR / "reports/final_model_report.md",
}

for name, path in paths.items():
    print(f"{name:20s} exists={path.exists()}  path={path}")

Working directory: c:\Users\Jeremy Chan\prediction-markets-phase1\notebooks
Base directory: c:\Users\Jeremy Chan\prediction-markets-phase1
markets_recent       exists=True  path=c:\Users\Jeremy Chan\prediction-markets-phase1\data\processed\markets_recent.parquet
features_mid         exists=True  path=c:\Users\Jeremy Chan\prediction-markets-phase1\data\processed\features_mid_recent_history_enriched.parquet
features_24h         exists=True  path=c:\Users\Jeremy Chan\prediction-markets-phase1\data\processed\features_24h_recent_history_enriched.parquet
offline_summary_csv  exists=True  path=c:\Users\Jeremy Chan\prediction-markets-phase1\reports\offline_snapshot_summary.csv
coefficients_csv     exists=True  path=c:\Users\Jeremy Chan\prediction-markets-phase1\reports\best_24h_coefficients.csv
poly_scored          exists=True  path=c:\Users\Jeremy Chan\prediction-markets-phase1\data\processed\current_polymarket_scored_trained.parquet
kalshi_scored        exists=True  path=c:\Users\Jeremy Chan

## Pipeline overview

The project pipeline is:

1. ingest raw market data
2. clean and normalise markets
3. build snapshot datasets
4. enrich snapshots with market-implied probabilities and history-derived features
5. evaluate offline models across snapshot timings
6. select the strongest model/snapshot combination
7. score current markets
8. review sanity checks and monitor reports

The key modelling question is:

**At what point in a market's lifecycle is the model most useful?**

In [2]:
markets_recent = pd.read_parquet(paths["markets_recent"])
features_mid = pd.read_parquet(paths["features_mid"])
features_24h = pd.read_parquet(paths["features_24h"])

print("markets_recent:", markets_recent.shape)
print("features_mid:", features_mid.shape)
print("features_24h:", features_24h.shape)

markets_recent: (470, 17)
features_mid: (470, 27)
features_24h: (446, 27)


In [3]:
markets_recent[[
    "market_id",
    "question",
    "category",
    "open_ts",
    "close_ts",
    "resolved_outcome",
    "volume",
    "liquidity",
]].head(10)

,market_id,question,category,open_ts,close_ts,resolved_outcome,volume,liquidity
0,255333,Will X remove likes + reposts counter in March?,None,2024-03-07 17:29:32.817000+00:00,2024-03-31 00:00:00+00:00,0,11403.704168,NaN
1,255335,Fed rate cut by September 18?,None,2024-03-07 18:53:28.276000+00:00,2024-09-18 00:00:00+00:00,1,20345317.826273,NaN
2,255336,Fed rate cut by November 7?,None,2024-03-07 18:53:28.351000+00:00,2024-11-07 00:00:00+00:00,1,2020982.792054,NaN
3,255337,Fed rate cut by December 18?,None,2024-03-07 18:53:28.352000+00:00,2024-12-18 00:00:00+00:00,1,2269863.269023,NaN
4,255338,Kung Fu Panda 4 over $48m opening weekend?,None,2024-03-07 18:57:06.892000+00:00,2024-03-11 00:00:00+00:00,1,43947.033092,0
5,255343,Congress passes bill banning TikTok by April 30?,None,2024-03-07 21:10:07.777000+00:00,2024-04-30 12:00:00+00:00,1,146220.701385,NaN
6,255344,TikTok sale announced in March?,None,2024-03-07 21:29:59.310000+00:00,2024-03-31 00:00:00+00:00,0,12293.848341,NaN
7,255345,Indian Election: Modi reelected?,None,2024-03-07 23:00:33.739000+00:00,2024-05-30 12:00:00+00:00,1,377942.060973,NaN
8,255346,Will a Democrat win North Carolina Governor Election?,None,2024-03-08 16:21:34.152000+00:00,2024-11-05 12:00:00+00:00,1,283661.747032,NaN
9,255347,Will a Republican win North Carolina Governor Election?,None,2024-03-08 16:21:34.332000+00:00,2024-11-05 12:00:00+00:00,0,545065.67519,NaN


## Snapshot logic

Each resolved market can be represented at multiple points in time:

- **open**: at market open
- **mid**: halfway between open and close
- **24h**: 24 hours before close

The intuition is that prediction difficulty changes over time:

- earlier snapshots have less information,
- later snapshots are closer to resolution and should be easier to model,
- a useful benchmark is whether the model beats the market-implied probability at the same timestamp.

In [4]:
features_24h[[
    "question",
    "snapshot_ts",
    "open_ts",
    "close_ts",
    "market_implied_prob",
    "resolved_outcome",
    "duration_hours"
]].head(10)

,question,snapshot_ts,open_ts,close_ts,market_implied_prob,resolved_outcome,duration_hours
0,Will X remove likes + reposts counter in March?,2024-03-30 00:00:00+00:00,2024-03-07 17:29:32.817000+00:00,2024-03-31 00:00:00+00:00,0.0550,0,558.507551
1,Fed rate cut by September 18?,2024-09-17 00:00:00+00:00,2024-03-07 18:53:28.276000+00:00,2024-09-18 00:00:00+00:00,0.9835,1,4661.108812
2,Fed rate cut by November 7?,2024-11-06 00:00:00+00:00,2024-03-07 18:53:28.351000+00:00,2024-11-07 00:00:00+00:00,0.9940,1,5861.108791
3,Fed rate cut by December 18?,2024-12-17 00:00:00+00:00,2024-03-07 18:53:28.352000+00:00,2024-12-18 00:00:00+00:00,0.9955,1,6845.108791
4,Kung Fu Panda 4 over $48m opening weekend?,2024-03-10 00:00:00+00:00,2024-03-07 18:57:06.892000+00:00,2024-03-11 00:00:00+00:00,0.9745,1,77.048086
5,Congress passes bill banning TikTok by April 30?,2024-04-29 12:00:00+00:00,2024-03-07 21:10:07.777000+00:00,2024-04-30 12:00:00+00:00,0.9880,1,1286.831173
6,TikTok sale announced in March?,2024-03-30 00:00:00+00:00,2024-03-07 21:29:59.310000+00:00,2024-03-31 00:00:00+00:00,0.0050,0,554.500192
7,Indian Election: Modi reelected?,2024-05-29 12:00:00+00:00,2024-03-07 23:00:33.739000+00:00,2024-05-30 12:00:00+00:00,0.9550,1,2004.990628
8,Will a Democrat win North Carolina Governor Election?,2024-11-04 12:00:00+00:00,2024-03-08 16:21:34.152000+00:00,2024-11-05 12:00:00+00:00,0.9550,1,5803.640513
9,Will a Republican win North Carolina Governor Election?,2024-11-04 12:00:00+00:00,2024-03-08 16:21:34.332000+00:00,2024-11-05 12:00:00+00:00,0.0385,0,5803.640463


## Offline snapshot summary

This is the cleanest offline result summary across snapshot timings.

Interpretation focus:
- rows used
- naive Brier
- market Brier
- model Brier
- Brier Skill Score
- whether the model beats the market

In [5]:
offline_summary = pd.read_csv(paths["offline_summary_csv"])
offline_summary

,snapshot,rows_used,naive_brier,market_brier,model_brier,bss_vs_naive,bss_vs_market,takeaway
0,open,0,NaN,NaN,NaN,NaN,NaN,no usable rows
1,mid,431,0.25,0.073034,0.066848,0.732606,0.084691,model beats market
2,24h,434,0.25,0.046290,0.036892,0.852431,0.203027,model beats market


## Offline takeaway

The main result is:

- **24h** is the strongest snapshot
- **mid** is weaker but still useful
- **open** is not currently robust enough in the current summary frame

This means the model adds the most value near resolution, especially at the 24h snapshot.

## Best 24h model coefficients

This view helps explain what is driving the strongest model.

Important questions:
- Is market-implied probability still the dominant signal?
- Do volume and category add useful information?
- Which extra features help or hurt?

In [6]:
coef_df = pd.read_csv(paths["coefficients_csv"])
coef_df.head(20)

,feature,coefficient,abs_coefficient
0,num__market_implied_prob,3.143656,3.143656
1,cat__category_fallback_Crypto,-0.957730,0.957730
2,cat__category_fallback_Sports,0.560604,0.560604
3,num__history_points_count,-0.408132,0.408132
4,num__log_volume,0.370494,0.370494
5,num__duration_hours,0.350855,0.350855
6,cat__category_fallback_Politics,0.248039,0.248039
7,cat__category_fallback_Other,0.215858,0.215858
8,num__prob_change_24h,-0.177562,0.177562
9,num__distance_from_0_5,0.157901,0.157901


## Coefficient interpretation

Key interpretation points:

- `market_implied_prob` remains the dominant feature
- `log_volume` contributes positively
- `duration_hours` contributes positively
- category effects matter
- some history-related features add limited or mixed value

So the model is not replacing the market — it is learning a correction layer on top of market probability plus a few structural/contextual features.

## Current Polymarket monitor

Polymarket is the cleaner live-monitor leg because:
- the model was trained on Polymarket historical data,
- it is closer to the training domain,
- sanity checks show fewer suspicious extremes.

In [7]:
poly = pd.read_parquet(paths["poly_scored"])
print(poly.shape)

poly.sort_values("abs_edge", ascending=False)[[
    "question",
    "category_fallback",
    "time_to_close_hours",
    "volume",
    "market_implied_prob",
    "model_prob",
    "model_minus_market",
    "abs_edge",
    "direction",
    "edge_bucket",
]].head(10)

(71, 22)


,question,category_fallback,time_to_close_hours,volume,market_implied_prob,model_prob,model_minus_market,abs_edge,direction,edge_bucket
0,Will the next Prime Minister of Hungary be Péter Magyar?,Other,623.109241,3490346.5433069677,0.6450,0.806144,0.161144,0.161144,bullish_vs_market,large
1,Will Sweden qualify for the 2026 FIFA World Cup?,Sports,623.109241,107805.19855699979,0.2750,0.125200,-0.149800,0.149800,bearish_vs_market,medium
2,Will Ukraine qualify for the 2026 FIFA World Cup?,Sports,623.109241,140899.94874300045,0.2700,0.123899,-0.146101,0.146101,bearish_vs_market,medium
3,BitBoy convicted?,Other,347.109241,107344.16603599946,0.1795,0.038830,-0.140670,0.140670,bearish_vs_market,medium
4,Will Italy qualify for the 2026 FIFA World Cup?,Sports,623.109241,222351.13500099987,0.6400,0.746258,0.106258,0.106258,bullish_vs_market,medium
5,Will Scottie Scheffler win the 2026 Masters tournament?,Sports,647.109241,93911.88471300004,0.1850,0.081458,-0.103542,0.103542,bearish_vs_market,medium
6,Will Poland qualify for the 2026 FIFA World Cup?,Sports,623.109241,461478.0072570008,0.3550,0.253001,-0.101999,0.101999,bearish_vs_market,medium
7,Will the next Prime Minister of Hungary be Viktor Orbán?,Other,623.109241,3044411.392079016,0.3450,0.248645,-0.096355,0.096355,bearish_vs_market,medium
8,Will the next Prime Minister of Hungary be István Kapitány?,Other,623.109241,8601447.815215958,0.0035,0.062474,0.058974,0.058974,bullish_vs_market,small
9,Will Rory McIlroy win the 2026 Masters tournament?,Sports,647.109241,71975.08676300007,0.0850,0.033968,-0.051032,0.051032,bearish_vs_market,small


## Current Kalshi monitor

Kalshi is integrated successfully, but should still be treated as more exploratory because:
- the current model was trained on Polymarket,
- Kalshi is a cross-venue application,
- sanity checks show many more suspicious extremes,
- confidence bands are needed to separate more credible rows from noisier ones.

In [8]:
kalshi = pd.read_parquet(paths["kalshi_scored"])
print(kalshi.shape)

kalshi["confidence_band"].value_counts(dropna=False)

(2303, 27)


confidence_band
exploratory        1389
high_confidence     914
Name: count, dtype: int64

In [9]:
kalshi.sort_values("abs_edge", ascending=False)[[
    "question",
    "category_fallback",
    "confidence_band",
    "time_to_close_hours",
    "volume_num",
    "open_interest_num",
    "market_implied_prob",
    "model_prob",
    "model_minus_market",
    "abs_edge",
]].head(15)

,question,category_fallback,confidence_band,time_to_close_hours,volume_num,open_interest_num,market_implied_prob,model_prob,model_minus_market,abs_edge
1389,"SOL price on Mar 20, 2026?",Crypto,high_confidence,64.266126,423.0,220.0,0.610,0.158788,-0.451212,0.451212
1390,"SOL price on Mar 20, 2026?",Crypto,high_confidence,64.266126,603.0,144.0,0.490,0.060976,-0.429024,0.429024
1391,"Ethereum price at Mar 20, 2026 at 5pm EDT?",Crypto,high_confidence,64.266126,970.0,361.0,0.620,0.196091,-0.423909,0.423909
0,"Will the WTI front-month settle oil price be >92.99 on Mar 18, 2026?",Macro/Commodities,exploratory,13.766126,14.0,14.0,0.510,0.088164,-0.421836,0.421836
1392,"SOL price on Mar 20, 2026?",Crypto,high_confidence,64.266126,501.0,232.0,0.660,0.240020,-0.419980,0.419980
1393,"SOL price on Mar 20, 2026?",Crypto,high_confidence,64.266126,1377.0,373.0,0.490,0.070755,-0.419245,0.419245
1394,"Ethereum price at Mar 20, 2026 at 5pm EDT?",Crypto,high_confidence,64.266126,2937.0,1352.0,0.525,0.107821,-0.417179,0.417179
1,What will any participating contestant say during Survivor Season 50 Episode 4?,Culture/Entertainment,exploratory,33.266126,44.0,21.0,0.500,0.095467,-0.404533,0.404533
2,"Will the silver open price be above $82.49 on Mar 20, 2026 at 5pm EDT?",Macro/Commodities,exploratory,64.266126,30.0,30.0,0.490,0.086280,-0.403720,0.403720
3,"SOL price on Mar 20, 2026?",Crypto,exploratory,64.266126,204.0,16.0,0.725,0.332098,-0.392902,0.392902


In [10]:
kalshi[kalshi["confidence_band"] == "high_confidence"].sort_values(
    "abs_edge", ascending=False
)[[
    "question",
    "category_fallback",
    "confidence_band",
    "time_to_close_hours",
    "volume_num",
    "open_interest_num",
    "market_implied_prob",
    "model_prob",
    "model_minus_market",
    "abs_edge",
]].head(10)

,question,category_fallback,confidence_band,time_to_close_hours,volume_num,open_interest_num,market_implied_prob,model_prob,model_minus_market,abs_edge
1389,"SOL price on Mar 20, 2026?",Crypto,high_confidence,64.266126,423.0,220.0,0.610,0.158788,-0.451212,0.451212
1390,"SOL price on Mar 20, 2026?",Crypto,high_confidence,64.266126,603.0,144.0,0.490,0.060976,-0.429024,0.429024
1391,"Ethereum price at Mar 20, 2026 at 5pm EDT?",Crypto,high_confidence,64.266126,970.0,361.0,0.620,0.196091,-0.423909,0.423909
1392,"SOL price on Mar 20, 2026?",Crypto,high_confidence,64.266126,501.0,232.0,0.660,0.240020,-0.419980,0.419980
1393,"SOL price on Mar 20, 2026?",Crypto,high_confidence,64.266126,1377.0,373.0,0.490,0.070755,-0.419245,0.419245
1394,"Ethereum price at Mar 20, 2026 at 5pm EDT?",Crypto,high_confidence,64.266126,2937.0,1352.0,0.525,0.107821,-0.417179,0.417179
1395,"SOL price on Mar 20, 2026?",Crypto,high_confidence,64.266126,261.0,184.0,0.425,0.032878,-0.392122,0.392122
1396,"Bitcoin price on Mar 20, 2026?",Crypto,high_confidence,64.266126,8339.0,4210.0,0.470,0.084933,-0.385067,0.385067
1397,"Ethereum price at Mar 20, 2026 at 5pm EDT?",Crypto,high_confidence,64.266126,1873.0,1556.0,0.435,0.050836,-0.384164,0.384164
1398,"Bitcoin price on Mar 20, 2026?",Crypto,high_confidence,64.266126,17116.0,10229.0,0.520,0.139287,-0.380713,0.380713


## Live-monitor interpretation

### Polymarket
- stronger trust level
- same-venue model application
- cleaner outputs
- fewer suspicious extremes

### Kalshi
- working monitor integration
- useful exploratory extension
- still noisier
- should be interpreted through `confidence_band`
- best used with stronger venue-specific hardening in a later phase

## Final artifacts produced

This project now produces:

- offline snapshot summary
- best 24h coefficient export
- current monitor report
- final model report
- current Polymarket scored parquet
- current Kalshi scored parquet

These artifacts support:
- model evaluation
- live monitor interpretation
- stakeholder walkthroughs
- next-phase hardening work

In [11]:
for report_path in [
    paths["offline_summary_csv"],
    paths["coefficients_csv"],
    paths["monitor_report"],
    paths["final_model_report"],
]:
    print(report_path, "exists=", report_path.exists())

c:\Users\Jeremy Chan\prediction-markets-phase1\reports\offline_snapshot_summary.csv exists= True
c:\Users\Jeremy Chan\prediction-markets-phase1\reports\best_24h_coefficients.csv exists= True
c:\Users\Jeremy Chan\prediction-markets-phase1\reports\current_monitor_report.md exists= True
c:\Users\Jeremy Chan\prediction-markets-phase1\reports\final_model_report.md exists= True


## Conclusions

### Main modelling conclusion
The **24h snapshot** is the strongest and most useful modelling point.

### Live-monitor conclusion
- **Polymarket** is the cleaner and more trustworthy live scorer.
- **Kalshi** is now integrated and filtered, but should still be treated as exploratory due to cross-venue domain shift.

### Most important project outcome
The system is no longer just an offline modelling exercise.
It now supports a real current-market monitoring workflow.

## Next steps
- stronger Kalshi-specific modelling
- stricter trust filters
- venue-specific history features
- more automated reporting / dashboarding